# Predictive AI Neural Seizure Feature Visualization

This notebook visualizes deterministic synthetic traces and feature curves for the Predictive AI Neural Seizure Analysis project. It is engineering evidence only and does not use patient data.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if cwd.name == 'notebooks':
    project_root = cwd.parent
elif (cwd / 'src' / 'neural_seizure_ai').exists():
    project_root = cwd
else:
    project_root = cwd / 'neural-seizure-ai-analysis'
sys.path.insert(0, str(project_root / 'src'))

from neural_seizure_ai.artifact_schema import validate_written_artifacts
from neural_seizure_ai.config import SimulationConfig
from neural_seizure_ai.export import export_student_to_c
from neural_seizure_ai.features import numeric_backend
from neural_seizure_ai.hil import benchmark_student
from neural_seizure_ai.pipeline import run_demo
from neural_seizure_ai.plots import write_plot_evidence

evidence_dir = project_root / 'docs' / 'evidence'
config = SimulationConfig(sensor='ecog', duration_seconds=90.0, preictal_start_seconds=55.0, ictal_start_seconds=75.0, seed=42)
print('numeric backend:', numeric_backend())

In [ ]:
result = run_demo(config, output_dir=evidence_dir)
plot_paths = write_plot_evidence(config, result, evidence_dir)
export_student_to_c(result.distillation, evidence_dir)
benchmark_student(result.feature_rows, result.distillation, output_dir=evidence_dir, target_label='notebook-host-reference')
validate_written_artifacts(evidence_dir)

print('windows:', result.window_count)
print('student sensitivity:', round(result.student_metrics.sensitivity, 3))
print('student specificity:', round(result.student_metrics.specificity, 3))
print('plots:', [path.name for path in plot_paths])

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(evidence_dir / 'synthetic-neural-ekg-traces.svg')))
display(SVG(filename=str(evidence_dir / 'biomarker-feature-curves.svg')))
display(SVG(filename=str(evidence_dir / 'feature-trajectories.svg')))

## Review Notes

- `synthetic-neural-ekg-traces.svg` shows the synthetic neural trace and BeagleBone AD8232-compatible EKG context.
- `biomarker-feature-curves.svg` isolates HFO ratio, PAC proxy, and connectivity over time.
- `feature-trajectories.svg` shows the broader feature set used by the teacher ensemble and distilled student.
- `demo-report.json`, `window-features.csv`, and `bbb-ekg-features.csv` are schema-validated by `artifact_schema.py`.
- `distilled_student.c` and `distilled_student.h` are generated for embedded review.